In [62]:
import itertools
import seaborn as sns
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import pandas as pd
from statsmodels.stats.anova import AnovaRM
from statsmodels.stats.multitest import multipletests
from scipy.stats import spearmanr
from pingouin import circ_corrcl
import statsmodels.api as sm
import statsmodels.formula.api as smf


In [63]:
def normalize_meanStd_column(df, column):
    df[column] = (df[column] - df[column].mean()) / df[column].std()


def fit_mixedEffects_model(df, formula, group_var):
    model = smf.mixedlm(
        formula,
        data=df,
        groups=df[group_var]
    ).fit()
    print(f"___________ {formula.split('~')[0].strip().capitalize()} model summary ___________")
    print(f" ")
    print(model.summary())
    print(f"\n")
    # return model


def column_asType(df, column, dtype):
    df[column] = df[column].astype(dtype)

In [64]:
def cohen_d(x, y):
    """Cohen's d for paired samples"""
    diff = x - y
    return np.mean(diff) / np.std(diff, ddof=1)

def run_paired_test(x, y, label):
    """
    Paired comparison between two dependent samples.
    Uses t-test if both normal, otherwise Wilcoxon signed-rank.
    
    Compares mean (or median) difference between two related measurements
    “Is accuracy for task=1 significantly higher than for task=0 for the same run?”
    p-value for difference
    Two related samples (often categorical condition)
    """
    normal_x = stats.shapiro(x).pvalue > 0.05
    normal_y = stats.shapiro(y).pvalue > 0.05

    if normal_x and normal_y:
        stat, pval = stats.ttest_rel(x, y)
        test_name = "Paired t-test"
    else:
        stat, pval = stats.wilcoxon(x, y)
        test_name = "Wilcoxon signed-rank"

    d = cohen_d(x, y)
    print(f"{label}: {test_name} → p={pval:.4f}, dCohen={d:.3f}")
    return pval

# def run_paired_test(x, y, label):
#     """Run paired test: t-test if normal, otherwise Wilcoxon signed-rank"""
#     normal_x = stats.shapiro(x).pvalue > 0.05
#     normal_y = stats.shapiro(y).pvalue > 0.05

#     if normal_x and normal_y:
#         stat, pval = stats.ttest_rel(x, y)
#         test_name = "Paired t-test"
#     else:
#         stat, pval = stats.wilcoxon(x, y)
#         test_name = "Wilcoxon signed-rank"

#     d = cohen_d(x, y)
#     print(f"{label}: {test_name} → p={pval:.4f}, dCohen={d:.3f}")
#     return pval, d

def significance_star(p):
    """Return significance stars based on p-value"""
    if p < 0.001: return "***"
    elif p < 0.01: return "**"
    elif p < 0.05: return "*"
    else: return "ns"

def add_significance_bar(ax, x1, x2, y, p, h=0.05, text_offset=0.02):
    """Draw significance bar with stars or 'ns'"""
    stars = significance_star(p)
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c='black')
    ax.text((x1+x2)/2, y+h+text_offset, stars, ha='center', va='bottom', fontsize=12)

def plot_bar_comparison(ax, group1, group2, labels, title, pval, y_label="Distance"):
    """Plot mean ± SEM with significance bar"""
    means = [np.mean(group1), np.mean(group2)]
    sems = [stats.sem(group1), stats.sem(group2)]
    x = np.arange(2)
    ax.bar(x, means, yerr=sems, capsize=6, color=["skyblue", "salmon"], alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel(y_label)
    ax.set_title(title)
    y_max = max(means) + max(sems) + 0.05
    add_significance_bar(ax, 0, 1, y_max, pval)
    ax.set_ylim(0, y_max + 0.15)

def remove_outliers(data):
    data = np.array(data)
    q1, q3 = np.percentile(data, [25, 75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    return data[(data >= lower) & (data <= upper)]

def compute_pd_pearson_Correlation(df, column_list, group=None):
    # linear correlations (values over 0.6 are considered strong)
    if group is not None:
        df.groupby('task')[column_list].corr()
    else:
        corr = df[column_list].corr()
    print(corr)

def compute_pd_spearman_correlation(df, column_list, group=None):
    # nonlinear correlations (e.g., monotonic relationships)
    if group is not None:
        df.groupby('task')[column_list].corr(method='spearman')
    else:
        corr = df[column_list].corr(method='spearman')
    print(corr)
    

In [65]:
normalize = []#['radius', 'angle']

In [66]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr

# === Helper functions ===

def corr_with_pvalues(df, cols=None, method='pearson'):
    """
    Returns a DataFrame with correlation coefficients and p-values.
    """
    if cols is None:
        cols = df.select_dtypes(include=[np.number]).columns
    
    corr_mat = pd.DataFrame(index=cols, columns=cols, dtype=float)
    pval_mat = pd.DataFrame(index=cols, columns=cols, dtype=float)

    for c1 in cols:
        for c2 in cols:
            x, y = df[c1].dropna(), df[c2].dropna()
            mask = x.index.intersection(y.index)
            if len(mask) > 2:
                if method == 'pearson':
                    r, p = pearsonr(df.loc[mask, c1], df.loc[mask, c2])
                else:
                    r, p = spearmanr(df.loc[mask, c1], df.loc[mask, c2])
                corr_mat.loc[c1, c2] = r
                pval_mat.loc[c1, c2] = p
            else:
                corr_mat.loc[c1, c2] = np.nan
                pval_mat.loc[c1, c2] = np.nan
    return corr_mat, pval_mat


def plot_correlation_heatmap(corr, pvals, alpha=0.05, title="Correlation Heatmap"):
    """
    Plots a heatmap masking non-significant correlations.
    """
    mask = pvals > alpha
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, annot=True, fmt=".2f", mask=mask, cmap="coolwarm", center=0, cbar_kws={'label': 'r'})
    plt.title(f"{title}\n(only significant at p<{alpha})", fontsize=14)
    plt.tight_layout()
    plt.show()


def plot_significant_pairs(df, corr, pvals, vars_x, vars_y, alpha=0.05):
    """
    Plots scatterplots only for significant variable pairs.
    """
    for x in vars_x:
        for y in vars_y:
            r, p = corr.loc[x, y], pvals.loc[x, y]
            if not np.isnan(r) and p < alpha:
                sns.lmplot(data=df, x=x, y=y, scatter_kws={'alpha':0.6})
                plt.title(f"{y} vs {x}\n r={r:.2f}, p={p:.3e}")
                plt.tight_layout()
                plt.show()


# user 2019-2020


In [ ]:
pathData = '/home/palatella/workspace/cybathlon_user/'
completeDataFrame = pd.read_csv(f"{pathData}completeResults_20192020.csv")

for col in normalize:   normalize_meanStd_column(completeDataFrame, col)

completeDataFrame['sin_angle'] = np.sin(completeDataFrame['angle'])
completeDataFrame['cos_angle'] = np.cos(completeDataFrame['angle'])

column_asType(completeDataFrame, 'task', 'category')
column_asType(completeDataFrame, 'band', 'category')

In [68]:
num_cols = completeDataFrame.select_dtypes(include='number').columns
num_cols = num_cols.drop(['runId'], errors='ignore')
# compute_pd_pearson_Correlation(completeDataFrame, num_cols)

In [ ]:
corr, pvals = corr_with_pvalues(completeDataFrame, 
                                cols=num_cols,
                                method='pearson')

# Display table with rounded values
display(corr.round(3))
display(pvals.round(3))

# Plot heatmap of significant correlations
plot_correlation_heatmap(corr, pvals, alpha=0.05, title="Pearson Correlation (significant only)")

# Optionally, also try Spearman:
corr_s, pvals_s = corr_with_pvalues(completeDataFrame, cols=num_cols, method='spearman')
plot_correlation_heatmap(corr_s, pvals_s, alpha=0.05, title="Spearman Correlation (significant only)")


In [70]:
# data = d_ToEye  # shape (band, sample, task)

# accuracy = run_accuracy  # shape (sample,)
# rejection = run_rejection  # shape (sample,)

# bands, samples, tasks = data.shape
# idx_division = np.sort(np.concatenate((stop_idx, rec_idx, [samples], [0])))

# boundaries = idx_division[idx_division>=0]
# segments = [range(start, end) for start, end in zip(boundaries[:-1], boundaries[1:])]

# segments_distance = np.empty((len(segments)), dtype=object)
# for i, seg in enumerate(segments):
#     segments_distance[i] = data[:, list(seg), :]

# seg_means = np.zeros((bands, len(segments), tasks))
# for b, t in itertools.product(range(bands), range(tasks)):
#     for i in range(0,len(segments_distance)):
#         seg_means[b, i, t] = np.mean(segments_distance[i][b, :, t])

In [71]:
# # ------------------------------
# # 1. Comparison between tasks (dependent) for each band
# # ------------------------------
# print("=== Task comparisons (paired, per band) ===")
# pvals_tasks = []
# for b in range(bands):
#     x, y = data[b, :, 0], data[b, :, 1]  # Task0 vs Task1
#     pval = run_paired_test(x, y, f"Band {b} (Task0 vs Task1)")
#     pvals_tasks.append(pval)

# # ------------------------------
# # 2. Comparison between bands (dependent) for each task
# # ------------------------------
# print("\n=== Band comparisons (paired, per task) ===")
# pvals_bands = []
# for t in range(tasks):
#     x, y = data[0, :, t], data[1, :, t]  # Band0 vs Band1
#     pval = run_paired_test(x, y, f"Task {t} (Band0 vs Band1)")
#     pvals_bands.append(pval)

# # ------------------------------
# # 3. Correlations with accuracy, rejection and CCC CVA calibration scores
# # ------------------------------
# print("\n=== Correlation with Accuracy, Rejection and CCC CVA calibrations ===")
# corr_results = []
# for b, t in itertools.product(range(bands), range(tasks)):
#     corr_acc, p_acc = stats.pearsonr(data[b, :, t], accuracy)
#     corr_rej, p_rej = stats.pearsonr(data[b, :, t], rejection)
#     # corr_cva, p_cva = stats.pearsonr(data[b, :, t], ccc_scores_cva_cal)

#     print(f"Band {b}, Task {t}: Corr_Accuracy={corr_acc:.3f} (p={p_acc:.4f}), "
#           f"Corr_Rejection={corr_rej:.3f} (p={p_rej:.4f}), ")
#         #   f"Corr_CCC_CVA={corr_cva:.3f} (p={p_cva:.4f})")
#     corr_results.append((corr_acc, p_acc, corr_rej, p_rej))#, corr_cva, p_cva))


# # ------------------------------
# # 4. Segment-level means
# # ------------------------------
# boundaries = idx_division[idx_division >= 0]
# segments = [range(start, end) for start, end in zip(boundaries[:-1], boundaries[1:])]
# segments_distance = np.empty((len(segments)), dtype=object)
# for i, seg in enumerate(segments):
#     segments_distance[i] = data[:, list(seg), :]

# seg_means = np.zeros((bands, len(segments), tasks))
# for b, t in itertools.product(range(bands), range(tasks)):
#     for i in range(len(segments_distance)):
#         seg_means[b, i, t] = np.mean(segments_distance[i][b, :, t])

# # ------------------------------
# # 5. Correlation between segment means across bands
# # ------------------------------
# print("\n=== Correlation between Band0 and Band1 segment means ===")
# for t in range(tasks):
#     corr, pval = stats.pearsonr(seg_means[0, :, t], seg_means[1, :, t])
#     print(f"Task {t}: Corr Band0 vs Band1 = {corr:.3f}, p={pval:.4f}")

# # Segment-level correlation with Accuracy/Rejection
# seg_acc = [np.mean(accuracy[list(seg)]) for seg in segments]
# seg_rej = [np.mean(rejection[list(seg)]) for seg in segments]

# print("\n=== Segment-level correlation with Accuracy and Rejection ===")
# for b, t in itertools.product(range(bands), range(tasks)):
#     corr_acc, p_acc = stats.pearsonr(seg_means[b, :, t], seg_acc)
#     corr_rej, p_rej = stats.pearsonr(seg_means[b, :, t], seg_rej)
#     print(f"Band {b}, Task {t}: Corr_SegAccuracy={corr_acc:.3f} (p={p_acc:.4f}), "
#           f"Corr_SegRejection={corr_rej:.3f} (p={p_rej:.4f})")

# # ------------------------------
# # 6. Bonferroni correction (dependent comparisons)
# # ------------------------------
# pvals = np.array(pvals_tasks + pvals_bands)
# pvals_corrected = np.minimum(pvals * len(pvals), 1.0)

# print("\n=== Bonferroni-corrected p-values (dependent) ===")
# for i, p in enumerate(pvals_corrected):
#     print(f"Comparison {i+1}: corrected p = {p:.4f}")


In [72]:
# # ------------------------------
# # 1. Task comparisons (paired) for each band
# # ------------------------------
# print("=== Task comparisons (paired) ===")
# pvals_tasks, ds_tasks = [], []
# for b in range(bands):
#     x, y = data[b, :, 0], data[b, :, 1]
#     pval, d = run_paired_test(x, y, f"Band {b} (Task0 vs Task1)")
#     pvals_tasks.append(pval)
#     ds_tasks.append(d)

# # ------------------------------
# # 2. Band comparisons (paired) for each task
# # ------------------------------
# print("\n=== Band comparisons (paired) ===")
# pvals_bands, ds_bands = [], []
# for t in range(tasks):
#     x, y = data[0, :, t], data[1, :, t]
#     pval, d = run_paired_test(x, y, f"Task {t} (Band0 vs Band1)")
#     pvals_bands.append(pval)
#     ds_bands.append(d)

# # ------------------------------
# # 3. Bonferroni correction (all dependent comparisons)
# # ------------------------------
# pvals = np.array(pvals_tasks + pvals_bands)
# pvals_corrected = np.minimum(pvals * len(pvals), 1.0)
# print("\n=== Bonferroni-corrected p-values ===")
# for i, p in enumerate(pvals_corrected):
#     print(f"Comparison {i+1}: corrected p = {p:.4f}")

# # ------------------------------
# # 4. Plot all comparisons (publication-style)
# # ------------------------------
# fig, axs = plt.subplots(2, 2, figsize=(10, 8))
# axs = axs.flatten()

# # Band 0: Task comparison
# plot_bar_comparison(axs[0], data[0, :, 0], data[0, :, 1],
#                     ["Task0", "Task1"], "Band 0: Task0 vs Task1", pvals_corrected[0])

# # Band 1: Task comparison
# plot_bar_comparison(axs[1], data[1, :, 0], data[1, :, 1],
#                     ["Task0", "Task1"], "Band 1: Task0 vs Task1", pvals_corrected[1])

# # Task 0: Band comparison
# plot_bar_comparison(axs[2], data[0, :, 0], data[1, :, 0],
#                     ["Band0", "Band1"], "Task 0: Band0 vs Band1", pvals_corrected[2])

# # Task 1: Band comparison
# plot_bar_comparison(axs[3], data[0, :, 1], data[1, :, 1],
#                     ["Band0", "Band1"], "Task 1: Band0 vs Band1", pvals_corrected[3])

# plt.suptitle("Mean ± SEM with Significance\n(* p<0.05, ** p<0.01, *** p<0.001, ns = not significant)", fontsize=14)
# plt.tight_layout(rect=[0, 0, 1, 0.95])
# plt.show()

In [73]:


# n_segments = len(segments_distance)
# bands, tasks = segments_distance[0].shape[0], segments_distance[0].shape[2]

# fig, axes = plt.subplots(bands, tasks, figsize=(10, 8), sharey=True)
# fig.suptitle("Repeated-Measures ANOVA + Paired Post-hoc Comparisons", fontsize=16, y=1.02)

# for b in range(bands):
#     for t in range(tasks):
#         ax = axes[b, t] if bands > 1 and tasks > 1 else axes[max(b, t)]

#         # Prepare data for repeated-measures analysis
#         data_segments = [remove_outliers(segments_distance[i][b, :, t]) for i in range(n_segments)]
#         n_runs = min(len(s) for s in data_segments)  # truncate if unequal

#         # Build long-format dataframe
#         df = pd.DataFrame({
#             'Run': np.tile(np.arange(n_runs), n_segments),
#             'Segment': np.repeat([f"Seg{i+1}" for i in range(n_segments)], n_runs),
#             'Value': np.concatenate([s[:n_runs] for s in data_segments])
#         })

#         # --- Repeated-measures ANOVA ---
#         aov = AnovaRM(df, depvar='Value', subject='Run', within=['Segment']).fit()
#         p_anova = aov.anova_table['Pr > F'][0]

#         # --- Post-hoc paired t-tests + Bonferroni correction ---
#         pairs = []
#         pvals = []
#         for i in range(n_segments - 1):
#             for j in range(i + 1, n_segments):
#                 tstat, pval = stats.ttest_rel(df[df['Segment'] == f"Seg{i+1}"]['Value'],
#                                               df[df['Segment'] == f"Seg{j+1}"]['Value'])
#                 pairs.append((i, j))
#                 pvals.append(pval)

#         reject, pvals_corr, _, _ = multipletests(pvals, method='bonferroni')

#         # --- Boxplot ---
#         sns.boxplot(data=data_segments, ax=ax, palette="pastel", width=0.6, showfliers=False)
#         # sns.stripplot(data=data_segments, ax=ax, color='black', size=4, jitter=True, alpha=0.6)

#         # --- Plot significance bars (staggered) ---
#         y_min, y_max = min([min(s) for s in data_segments]), max([max(s) for s in data_segments])
#         y_base = y_max * 1.05
#         y_step = (y_max - y_min) * 0.15

#         for (k, (i, j)) in enumerate(pairs):
#             if not reject[k]:
#                 continue
#             dist = abs(i - j)
#             star = significance_star(pvals_corr[k])
#             y = y_base + (dist - 1) * y_step
#             ax.plot([i, i, j, j], [y, y + y_step*0.15, y + y_step*0.15, y], lw=1.5, c='k')
#             ax.text((i + j)/2, y + y_step*0.17, star, ha='center', va='bottom', fontsize=11)

#         ax.set_xticks(range(n_segments))
#         ax.set_xticklabels([f"Seg {i+1}" for i in range(n_segments)])
#         ax.set_title(f"Band {b}, Task {t}\nANOVA p={p_anova:.3e}")
#         ax.set_xlabel("Segments")
#         ax.set_ylabel("Distance")
#         sns.despine(ax=ax)

# plt.tight_layout()
# plt.show()


In [74]:
# # --- Spearman correlations for radius ---
# rho_acc, p_acc = spearmanr(completeDataFrame['radius'], completeDataFrame['accuracy'])
# rho_rej, p_rej = spearmanr(completeDataFrame['radius'], completeDataFrame['rejection'])

# print(f"Radius-Accuracy: rho = {rho_acc:.3f}, p = {p_acc:.3f}")
# print(f"Radius-Rejection: rho = {rho_rej:.3f}, p = {p_rej:.3f}")

# # --- Circular-linear correlations for angle ---
# r_acc, p_acc = circ_corrcl(completeDataFrame['angle'], completeDataFrame['accuracy'])
# r_rej, p_rej = circ_corrcl(completeDataFrame['angle'], completeDataFrame['rejection'])

# print(f"Angle-Accuracy: r = {r_acc:.3f}, p = {p_acc:.3f}")
# print(f"Angle-Rejection: r = {r_rej:.3f}, p = {p_rej:.3f}")

In [75]:
# # # Adjust proportions to avoid exactly 0 or 1
# # completeDataFrame['accuracy_adj'] = (completeDataFrame['accuracy'] * (len(completeDataFrame) - 1) + 0.5) / len(completeDataFrame)
# # completeDataFrame['rejection_adj'] = (completeDataFrame['rejection'] * (len(completeDataFrame) - 1) + 0.5) / len(completeDataFrame)

# # Model accuracy
# model_acc = smf.glm(
#     formula="accuracy ~ radius + sin_angle + cos_angle + task * band",
#     data=completeDataFrame, 
#     family=sm.families.Binomial()
# ).fit()

# # Model rejection
# model_rej = smf.glm(
#     formula="rejection ~ radius + sin_angle + cos_angle + task * band",
#     data=completeDataFrame, 
#     family=sm.families.Binomial()
# ).fit()

# print(model_acc.summary())
# print(model_rej.summary())


In [76]:
# plt.figure()
# # Radius vs accuracy
# sns.scatterplot(x='radius', y='accuracy', data=completeDataFrame)
# sns.scatterplot(x='radius', y='rejection', data=completeDataFrame)
# plt.show()

# plt.figure()
# # Angle vs accuracy
# sns.scatterplot(x=np.degrees(completeDataFrame['angle']), y='accuracy', data=completeDataFrame)
# sns.scatterplot(x=np.degrees(completeDataFrame['angle']), y='rejection', data=completeDataFrame)
# plt.show()


In [77]:
# # Define variance components for nested random effects
# vc = {"runId": "0 + C(runId)"}  # random intercept per runId

# # Mixed-effects model for accuracy
# model_accuracy = smf.mixedlm(
#     "accuracy ~ task + band + radius + sin_angle + cos_angle",
#     data=completeDataFrame,
#     groups=completeDataFrame["day"],  # top-level random intercept per day
#     vc_formula=vc
# )

# result_accuracy = model_accuracy.fit(reml=True)
# print(result_accuracy.summary())

# # Mixed-effects model for rejection
# model_rejection = smf.mixedlm(
#     "rejection ~ task + band + radius + sin_angle + cos_angle",
#     data=completeDataFrame,
#     groups=completeDataFrame["day"],
#     vc_formula=vc
# )

# result_rejection = model_rejection.fit(reml=True)
# print(result_rejection.summary())

# fit_mixedEffects_model(completeDataFrame, "accuracy ~ task * band + radius + sin_angle + cos_angle", group_var="day")
# fit_mixedEffects_model(completeDataFrame, "rejection ~ task * band + radius + sin_angle + cos_angle", group_var="day")


In [78]:
# ols_acc = smf.ols("rejection ~ radius + sin_angle  + C(task)*C(band)", data=completeDataFrame).fit()
# print(ols_acc.summary())

# user 2023-2024

In [ ]:
pathData = '/home/palatella/workspace/cybathlon_user/'
completeDataFrame2 = pd.read_csv(f"{pathData}completeResults_20232024.csv")

for col in normalize:   normalize_meanStd_column(completeDataFrame2, col)

completeDataFrame2['sin_angle'] = np.sin(completeDataFrame2['angle'])
completeDataFrame2['cos_angle'] = np.cos(completeDataFrame2['angle'])

column_asType(completeDataFrame2, 'task', 'category')
column_asType(completeDataFrame2, 'band', 'category')
# column_asType(completeDataFrame2, 'runId', 'category')

fit_mixedEffects_model(completeDataFrame2, "accuracy ~ radius  + sin_angle + task * band", group_var="day")
fit_mixedEffects_model(completeDataFrame2, "rejection ~ radius  + sin_angle + task * band", group_var="day")

/home/palatella/workspace/pytorch_env/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


___________ Accuracy model summary ___________
 
            Mixed Linear Model Regression Results
Model:                MixedLM   Dependent Variable:   accuracy
No. Observations:     404       Method:               REML    
No. Groups:           42        Scale:                0.0038  
Min. group size:      4         Log-Likelihood:       472.9750
Max. group size:      24        Converged:            Yes     
Mean group size:      9.6                                     
--------------------------------------------------------------
                    Coef.  Std.Err.   z    P>|z| [0.025 0.975]
--------------------------------------------------------------
Intercept            0.403    0.090  4.471 0.000  0.226  0.579
task[T.1]           -0.000    0.009 -0.013 0.989 -0.017  0.017
band[T.1]            0.002    0.009  0.246 0.805 -0.015  0.020
task[T.1]:band[T.1] -0.000    0.012 -0.019 0.985 -0.024  0.024
radius               0.076    0.026  2.866 0.004  0.024  0.127
sin_angle          

/home/palatella/workspace/pytorch_env/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
